# W10 — Assignment notebook

**Tema:** Particionamiento de datos + Partition Pruning  
**Dataset:** `silver_planet_v3` (6 087 planetas — NASA Exoplanet Archive)

## Setup

In [ ]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path(".").resolve()
DB_PATH      = PROJECT_ROOT / "data" / "exoplanets.duckdb"
RAW_CSV      = PROJECT_ROOT / "data" / "raw" / "pscomppars.csv"
PART_DIR     = PROJECT_ROOT / "data" / "partitioned"

def sql_path(p: Path) -> str:
    return "'" + p.resolve().as_posix().replace("'", "''") + "'"

con = duckdb.connect(str(DB_PATH))

con.execute("DROP VIEW IF EXISTS raw_ps")
con.execute(f"CREATE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_path(RAW_CSV)})")

PART_DIR.mkdir(parents=True, exist_ok=True)
print("Setup OK — filas en silver_planet_v3:",
      con.sql("SELECT COUNT(*) FROM silver_planet_v3").fetchone()[0])

## 1. Evidencia de particionamiento

Particionamos `silver_planet_v3` por `disc_era` exportando un archivo Parquet por era.

In [ ]:
# Exportar una partición Parquet por disc_era
eras = ['pre-2000', '2000s', '2010s', '2020s', 'unknown']

for era in eras:
    out = PART_DIR / f"disc_era={era}"
    out.mkdir(exist_ok=True)
    con.execute(f"""
        COPY (SELECT * FROM silver_planet_v3 WHERE disc_era = '{era}')
        TO '{out}/data.parquet' (FORMAT PARQUET)
    """)

print("Particiones generadas:")
for p in sorted(PART_DIR.rglob('*.parquet')):
    size = p.stat().st_size
    print(f"  {p.relative_to(PART_DIR)}  →  {size:,} bytes  ({size/1024:.1f} KB)")

## 2. Número de archivos generados

In [ ]:
archivos = list(PART_DIR.rglob('*.parquet'))
print(f"Total archivos Parquet generados: {len(archivos)}")
for f in sorted(archivos):
    print(f"  {f.name}  en  {f.parent.name}/")

## 3. Resumen por partición

In [ ]:
print(f"{'disc_era':<12}  {'rows':>6}  {'size_KB':>8}  {'pct_rows':>9}")
print("-" * 42)
totals = {'rows': 0, 'size': 0}
for era in eras:
    f = PART_DIR / f"disc_era={era}" / "data.parquet"
    rows = con.sql(f"SELECT COUNT(*) FROM '{f}'").fetchone()[0]
    size = f.stat().st_size / 1024
    pct  = rows / 6087 * 100
    print(f"  {era:<12}  {rows:>6}  {size:>7.1f}  {pct:>8.2f}%")
    totals['rows'] += rows
print("-" * 42)
print(f"  {'TOTAL':<12}  {totals['rows']:>6}")

## 4. Evidencia de pruning

Comparamos el plan de ejecución de una query sobre la tabla completa vs. sobre una sola partición.

In [ ]:
# Sin pruning — escanea las 6087 filas
print("=== SIN pruning — TABLE_SCAN sobre silver_planet_v3 (6087 filas) ===")
con.sql("""
    EXPLAIN ANALYZE
    SELECT discoverymethod_canon, COUNT(*) AS n
    FROM silver_planet_v3
    GROUP BY discoverymethod_canon
    ORDER BY n DESC
""").show()

# Con pruning — lee solo la partición 2010s (3683 filas)
print("=== CON pruning — READ_PARQUET sobre disc_era=2010s (3683 filas) ===")
con.sql(f"""
    EXPLAIN ANALYZE
    SELECT discoverymethod_canon, COUNT(*) AS n
    FROM read_parquet('{PART_DIR}/disc_era=2010s/data.parquet')
    GROUP BY discoverymethod_canon
    ORDER BY n DESC
""").show()

## 5. Guardado de EXPLAIN ANALYZE como texto

In [ ]:
out_txt = PROJECT_ROOT / "docs" / "w10b_explain_analyze_pruning.txt"
out_txt.parent.mkdir(exist_ok=True)

lines = []
lines.append("=== SIN PRUNING — silver_planet_v3 (6087 filas) ===\n")
rows_full = con.sql("""
    EXPLAIN ANALYZE
    SELECT discoverymethod_canon, COUNT(*) AS n
    FROM silver_planet_v3
    GROUP BY discoverymethod_canon ORDER BY n DESC
""").fetchall()
for r in rows_full:
    lines.append(r[1] + "\n")

lines.append("\n=== CON PRUNING — disc_era=2010s (3683 filas) ===\n")
rows_prune = con.sql(f"""
    EXPLAIN ANALYZE
    SELECT discoverymethod_canon, COUNT(*) AS n
    FROM read_parquet('{PART_DIR}/disc_era=2010s/data.parquet')
    GROUP BY discoverymethod_canon ORDER BY n DESC
""").fetchall()
for r in rows_prune:
    lines.append(r[1] + "\n")

out_txt.write_text("".join(lines))
print(f"Guardado en: {out_txt}")

## 6. Explicación del filtro de pruning usado

El filtro usado es **`disc_era = '2010s'`**, que corresponde a planetas descubiertos entre 2010 y 2019.

Cuando el motor lee `read_parquet('disc_era=2010s/data.parquet')` solo abre ese archivo, **saltándose completamente** los otros 4 (pre-2000, 2000s, 2020s, unknown). El plan EXPLAIN ANALYZE lo confirma:

- **Sin pruning:** `TABLE_SCAN → 6,087 rows`  
- **Con pruning:** `READ_PARQUET → Total Files Read: 1 → 3,683 rows`

Reducción de datos escaneados: **39.5%** (se evitan 2 404 filas).

## 7. Decisión de partición

In [ ]:
# Distribución de disc_era
print("Distribución por disc_era (columna de partición elegida):")
con.sql("""
    SELECT disc_era,
           COUNT(*) AS n,
           ROUND(COUNT(*)*100.0/6087, 2) AS pct
    FROM silver_planet_v3
    GROUP BY disc_era ORDER BY disc_era
""").show()

# Distribución alternativa por discoverymethod_canon
print("\nAlternativa — distribución por discoverymethod_canon:")
con.sql("""
    SELECT discoverymethod_canon,
           COUNT(*) AS n,
           ROUND(COUNT(*)*100.0/6087, 2) AS pct
    FROM silver_planet_v3
    GROUP BY discoverymethod_canon ORDER BY n DESC
""").show()

## 8. ¿Por qué `disc_era` sí?

`disc_era` es una buena columna de partición porque:

1. **Cardinalidad baja y fija:** solo 5 valores posibles (pre-2000, 2000s, 2010s, 2020s, unknown). Esto genera exactamente 5 archivos — no hay riesgo de explosión.
2. **Distribución razonable:** la partición más grande (2010s) tiene 3 683 filas; la más pequeña con datos reales (pre-2000) tiene 30. Hay skew, pero manejable.
3. **Patrón de acceso real:** analistas que estudian tendencias temporales filtran frecuentemente por era: *¿cómo cambió la tasa de descubrimiento en la década Kepler?*
4. **Columna derivada y estable:** `disc_era` no cambiará para registros históricos — las particiones son inmutables en la práctica.

**Limitación:** hay skew notable (60.5% en 2010s vs 0.49% en pre-2000). En un sistema de producción con TB de datos, esto podría requerir sub-particionamiento de 2010s.

## 9. ¿Qué otra columna evaluaría?

Evaluaría **`discoverymethod_canon`** como segunda opción, pero la descartaría por las razones del punto 10.

Una alternativa viable sería una **partición compuesta** `disc_era / discoverymethod_canon` para queries que filtran por ambas dimensiones (ej. *¿cuántos planetas de tránsito se descubrieron en los 2010s?*). Sin embargo, con 5 × 11 = 55 posibles combinaciones y solo 6 087 filas totales, la mayoría de los archivos serían minúsculos.

## 10. Riesgo de *small files*

In [ ]:
# Si particionáramos por discoverymethod_canon:
print("Riesgo de small files — partición hipotética por discoverymethod_canon:")
con.sql("""
    SELECT discoverymethod_canon,
           COUNT(*) AS n_rows,
           ROUND(COUNT(*)*100.0/6087, 2) AS pct,
           CASE WHEN COUNT(*) < 50 THEN '⚠ SMALL FILE' ELSE 'OK' END AS riesgo
    FROM silver_planet_v3
    GROUP BY discoverymethod_canon
    ORDER BY n_rows ASC
""").show()

Si particionáramos por `discoverymethod_canon`, **7 de 11 particiones** tendrían menos de 50 filas. En un sistema distribuido (Spark, Hive, S3+Athena), cada archivo implica overhead de apertura y metadata. Archivos de 1–9 filas son prácticamente todo overhead y casi cero dato útil.

Con `disc_era` solo `pre-2000` (30 filas, 5.2 KB) y `unknown` (1 fila, 2.3 KB) son pequeños, pero son marginales y no afectan el rendimiento general.

## 11. Reflexión breve

El particionamiento es una decisión de diseño físico que debe derivarse del **patrón de acceso esperado**, no de la estructura del dato. Las preguntas correctas son:

- *¿Qué columnas aparecen más frecuentemente en el `WHERE`?*
- *¿Cuántas particiones genera la cardinalidad de esa columna?*
- *¿El tamaño de cada partición justifica el overhead de abrirla como archivo separado?*

Con 6 087 filas, `disc_era` es la única columna que pasa los tres filtros razonablemente. En un dataset real de millones de exoplanetas, la decisión sería más matizada.

## 12. ¿Cuándo particionar ayuda?

Particionar ayuda cuando:

- El dataset tiene **muchos GB o TB** y las queries filtran consistentemente por la columna de partición.
- La cardinalidad de la columna de partición es **baja y estable** (fecha por año/mes, región, categoría).
- Las particiones tienen **tamaño similar** (no hay skew extremo).
- Se necesita **purgar o reemplazar datos** de un rango específico (ej. repartir solo el mes actual sin tocar el histórico).
- El sistema de almacenamiento cobra por datos escaneados (ej. Athena, BigQuery): el pruning reduce costos directamente.

## 13. ¿Cuándo particionar empeora el diseño?

Particionar empeora el diseño cuando:

- El dataset es **pequeño** (como aquí: 6 087 filas caben en RAM con margen). El overhead de abrir múltiples archivos supera el beneficio del pruning.
- La columna de partición tiene **alta cardinalidad** (ej. `hostname_canon` — miles de valores únicos → miles de archivos minúsculos).
- Hay **skew severo**: una partición acapara el 95% de los datos y el resto son archivos de 1 fila. El motor sigue leyendo casi todo.
- Las queries **nunca filtran por la columna de partición**: el pruning no se activa y se paga el overhead sin beneficio.
- Se usa **partición compuesta** con demasiadas dimensiones, generando una explosión combinatoria de archivos vacíos o casi vacíos.